## Determinação do tamanho de janela \(L_M\)

Esta seção define o tamanho da janela deslizante \(L_M\) usado na construção do TVG.
Foram considerados **dois critérios** complementares:

1. **Comparação de arestas** (inspirado em Rosário, 2017): número médio de arestas \(N_a\)
   da rede original versus rede com *motifs* embaralhados no tempo, em função de \(L_M\).
   Identifica o menor \(L_M\) a partir do qual a rede original se distancia da nula
   (piso metodológico).
2. **Critério fisiológico**: ancora \(L_M\) à escala temporal das bandas EEG
   (em especial ~1 ciclo alfa, via IAF), tipicamente da ordem de **100 ms**.

O valor adotado no pipeline é \(L_M\) correspondente a **100 ms** (25 amostras a 250 Hz).


### Imports

In [62]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mne
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Cursor/VS Code: evita erro de nbformat em sessões antigas do kernel
pio.renderers.default = "vscode"
try:
    import nbformat as _nbformat
    import plotly.io._renderers as _plotly_renderers

    _plotly_renderers.nbformat = _nbformat
except Exception:
    pass

from src.graph.aggregate import (
    aggregate_static_network,
    edges_per_time,
    temporal_statistics,
)
from src.graph.eeg_crop import crop_continuous_segment
from src.graph.motifs import trans_motifs
from src.graph.network import graph_metrics
from src.graph import threshold as _threshold_mod
import importlib

importlib.reload(_threshold_mod)
estimate_threshold = _threshold_mod.estimate_threshold
from src.graph.tvg import build_tvg_parallel


### Paths

In [63]:
SOURCE_DIR = Path("../../data/preprocessed")

print("SOURCE_DIR:", SOURCE_DIR.resolve())

SOURCE_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\preprocessed


### Parâmetros

In [64]:
# Recorte contínuo comum (mesmo critério do pipeline 01_build_networks)
COMMON_JSON = Path("../../data/intermediate/tvg_common_continuous_summary.json")
_common = json.loads(COMMON_JSON.read_text(encoding="utf-8"))
TVG_T0 = float(_common["t0"])
TVG_T1 = float(_common["t1"])
N_SEC = int(_common.get("N_SEC_sugerido", 8))
WINDOW_SIZES = list(range(5, 26))  # L_M em amostras (como na figura de Rosário)
WINDOW_SIZES_ML = [8, 10, 12, 15, 20, 25]  # grade reduzida para downstream
MAX_LAG_CAP = 5  # lag máximo em amostras; por janela: min(MAX_LAG_CAP, L_M - 1)
N_SHUFFLES = 1
THRESHOLD_PERCENTILE = 99.0
RANDOM_STATE = 42
N_SUBJECTS_ML = 3  # pares EO/EC no método downstream


### Listar arquivos e escolher 1 EEG

In [65]:
files = sorted(SOURCE_DIR.glob("sub-*.set"))
print(f"Encontrados {len(files)} arquivos")
files[:5]

Encontrados 315 arquivos


[WindowsPath('../../data/preprocessed/sub-010060_EO.set'),
 WindowsPath('../../data/preprocessed/sub-010061_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010061_EO.set'),
 WindowsPath('../../data/preprocessed/sub-010062_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010062_EO.set')]

In [66]:
file = files[0]
file

WindowsPath('../../data/preprocessed/sub-010060_EO.set')

### Carregar EEG e motifs

In [67]:
# ==========================
# LOAD EEG
# ==========================
raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)
raw.pick_types(eeg=True)

eeg, sfreq, start_samp, stop_samp = crop_continuous_segment(
    raw,
    TVG_T0,
    TVG_T1,
    N_SEC,
    file_name=file.name,
)

print(f"Arquivo: {file.name}")
print(
    f"Shape EEG: {eeg.shape} | Fs: {sfreq} Hz | "
    f"Recorte: [{start_samp/sfreq:.3f}, {stop_samp/sfreq:.3f}] s "
    f"(N_SEC={N_SEC}, comum [{TVG_T0:.3f}, {TVG_T1:.3f}])"
)

# ==========================
# MOTIFS
# ==========================
motifs = trans_motifs(eeg)
print(f"Shape motifs: {motifs.shape}")


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Arquivo: sub-010060_EO.set
Shape EEG: (61, 1250) | Fs: 250.0 Hz | Duração: 5 s
Shape motifs: (61, 1248)


C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\323120992.py:4: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\323120992.py:4: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.



### Critério 1 — Comparação de arestas (original vs embaralhada)

Para cada tamanho de janela: estima o limiar (shuffle da série + p99), constrói o TVG original
e o TVG com *motifs* embaralhados no tempo (mesmo limiar), e calcula o \(N_a\) médio.

**Atenção:** cada \(L_M\) roda 2 TVGs — pode demorar vários minutos.


In [68]:
rng = np.random.default_rng(RANDOM_STATE)

rows = []

for L_M in tqdm(WINDOW_SIZES, desc="L_M"):
    max_lag = min(MAX_LAG_CAP, L_M - 1)

    # ==========================
    # THRESHOLD (tese: shuffle serie + p99)
    # ==========================
    threshold_info = estimate_threshold(
        eeg,
        L_M,
        max_lag,
        n_shuffles=N_SHUFFLES,
        percentile=THRESHOLD_PERCENTILE,
        random_state=RANDOM_STATE,
    )
    threshold = threshold_info["threshold"]

    # ==========================
    # TVG ORIGINAL
    # ==========================
    tvg_orig = build_tvg_parallel(
        motifs=motifs,
        window_size=L_M,
        max_lag=max_lag,
        threshold=threshold,
    )
    na_orig = float(np.mean(edges_per_time(tvg_orig)))

    # ==========================
    # MOTIFS EMBARALHADOS + TVG
    # ==========================
    motifs_shuf = motifs.copy()
    for ch in range(motifs_shuf.shape[0]):
        rng.shuffle(motifs_shuf[ch])

    tvg_shuf = build_tvg_parallel(
        motifs=motifs_shuf,
        window_size=L_M,
        max_lag=max_lag,
        threshold=threshold,
    )
    na_shuf = float(np.mean(edges_per_time(tvg_shuf)))

    rows.append(
        {
            "L_M": L_M,
            "L_M_ms": L_M / sfreq * 1000,
            "max_lag": max_lag,
            "threshold": threshold,
            "Na_original": na_orig,
            "Na_embaralhado": na_shuf,
        }
    )
    print(
        f"L_M={L_M:2d} pts ({L_M / sfreq * 1000:.1f} ms) | "
        f"thr={threshold:.4f} | Na_orig={na_orig:.1f} | Na_shuf={na_shuf:.1f}"
    )

df_na = pd.DataFrame(rows)
df_na


L_M:   0%|          | 0/21 [00:00<?, ?it/s]

L_M:   5%|▍         | 1/21 [00:47<15:42, 47.12s/it]

L_M= 5 pts (20.0 ms) | thr=1.0000 | Na_orig=789.2 | Na_shuf=782.0


L_M:  10%|▉         | 2/21 [01:37<15:35, 49.25s/it]

L_M= 6 pts (24.0 ms) | thr=1.0000 | Na_orig=775.9 | Na_shuf=783.1


...

L_M:  95%|█████████▌| 20/21 [17:40<00:51, 51.25s/it]

L_M=24 pts (96.0 ms) | thr=0.5263 | Na_orig=1031.2 | Na_shuf=357.2


L_M: 100%|██████████| 21/21 [18:34<00:00, 53.06s/it]

L_M=25 pts (100.0 ms) | thr=0.5238 | Na_orig=992.5 | Na_shuf=338.6


,L_M,L_M_ms,max_lag,threshold,Na_original,Na_embaralhado
0,5,20.0,4,1.000000,789.179984,781.963680
1,6,24.0,5,1.000000,775.902991,783.136621
2,7,28.0,5,1.000000,555.615696,291.018608
3,8,32.0,5,1.000000,400.199190,104.497166
4,9,36.0,5,1.000000,292.362237,36.385737
5,10,40.0,5,0.857143,379.409570,32.846715
6,11,44.0,5,0.833333,362.909091,58.460227
7,12,48.0,5,0.750000,572.677498,85.632006
8,13,52.0,5,0.750000,552.426016,76.028455
9,14,56.0,5,0.700000,614.723352,91.916192


### Gráfico \(N_a\) vs \(L_M\)


In [69]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_na["L_M"],
        y=df_na["Na_original"],
        mode="lines+markers",
        name="Redes originais",
        marker=dict(symbol="square", size=8, color="black"),
        line=dict(color="black", width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=df_na["L_M"],
        y=df_na["Na_embaralhado"],
        mode="lines+markers",
        name="Redes embaralhadas",
        marker=dict(symbol="circle", size=8, color="crimson"),
        line=dict(color="crimson", width=2),
    )
)

fig.update_layout(
    title=(
        f"Número médio de arestas vs tamanho da janela — {file.stem}<br>"
        "<sup>Comparação redes originais vs embaralhadas (Rosário, 2017)</sup>"
    ),
    xaxis_title="Tamanho da janela L_M (pts)",
    yaxis_title="N_a (número médio de arestas)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.02, y=0.98),
    height=500,
)

display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


### Interpretação (critério 1)

Menor \(L_M\) em que a rede original se separa da embaralhada
(critério: \(N_a^{\text{orig}} > 2 \cdot N_a^{\text{shuf}}\) e diferença absoluta \(\geq 10\) arestas).
Esse valor é um **piso**: janelas menores tendem a não se distinguir do acaso; o valor
operacional final de \(L_M\) é definido em conjunto com o critério fisiológico.


In [70]:
sep = (
    (df_na["Na_original"] > 2 * df_na["Na_embaralhado"])
    & ((df_na["Na_original"] - df_na["Na_embaralhado"]) >= 10)
)

if sep.any():
    row = df_na.loc[sep].iloc[0]
    L_star = int(row["L_M"])
    L_star_ms = float(row["L_M_ms"])
    print(f"Menor L_M com separação clara: {L_star} pts (~{L_star_ms:.1f} ms a {sfreq:.0f} Hz)")
    print(
        f"Na_original={row['Na_original']:.1f} | "
        f"Na_embaralhado={row['Na_embaralhado']:.1f}"
    )
    print(
        f"\nPara o pipeline (WINDOW_MS), use aproximadamente: "
        f"WINDOW_MS = {int(round(L_star_ms))} ms"
    )
else:
    print(
        "Nenhum L_M atendeu o critério de separação nesta faixa. "
        "Inspecione o gráfico ou ajuste N_SEC / surrogates / critério."
    )

df_na[["L_M", "L_M_ms", "Na_original", "Na_embaralhado"]]

Menor L_M com separação clara: 8 pts (~32.0 ms a 250 Hz)
Na_original=400.2 | Na_embaralhado=104.5

Para o pipeline (WINDOW_MS), use aproximadamente: WINDOW_MS = 32 ms


,L_M,L_M_ms,Na_original,Na_embaralhado
0,5,20.0,789.179984,781.963680
1,6,24.0,775.902991,783.136621
2,7,28.0,555.615696,291.018608
3,8,32.0,400.199190,104.497166
4,9,36.0,292.362237,36.385737
5,10,40.0,379.409570,32.846715
6,11,44.0,362.909091,58.460227
7,12,48.0,572.677498,85.632006
8,13,52.0,552.426016,76.028455
9,14,56.0,614.723352,91.916192


### Critério 2 — Fisiológico (faixa de bandas EEG)

Converte \(L_M\) em milissegundos e compara com períodos típicos de bandas EEG
(theta ~125 ms, alpha ~100 ms, beta ~50 ms). A faixa compatível com alpha/beta ancora
a interpretação do critério de arestas e antecipa a análise da IAF (~1 ciclo alfa).


In [71]:
# ==========================
# FISIOLÓGICO
# ==========================
BAND_PERIOD_MS = {
    "beta (~20 Hz)": 50.0,
    "alpha (~10 Hz)": 100.0,
    "theta (~8 Hz)": 125.0,
}

rows_phys = []
for L_M in WINDOW_SIZES:
    ms = L_M / sfreq * 1000
    if ms < 50:
        faixa = "curta demais (< ~1 ciclo beta)"
    elif ms <= 150:
        faixa = "compatível com alpha/beta (~50–150 ms)"
    else:
        faixa = "mais longa (> 1.5 ciclo alpha)"
    rows_phys.append(
        {
            "L_M": L_M,
            "L_M_ms": ms,
            "frac_alpha": ms / BAND_PERIOD_MS["alpha (~10 Hz)"],
            "frac_beta": ms / BAND_PERIOD_MS["beta (~20 Hz)"],
            "faixa": faixa,
        }
    )

df_phys = pd.DataFrame(rows_phys)
L_phys_min = int(df_phys.loc[df_phys["L_M_ms"] >= 50, "L_M"].min())
L_phys_max = int(df_phys.loc[df_phys["L_M_ms"] <= 150, "L_M"].max())
print(
    f"Faixa fisiológica sugerida: L_M = {L_phys_min}–{L_phys_max} pts "
    f"({L_phys_min / sfreq * 1000:.0f}–{L_phys_max / sfreq * 1000:.0f} ms a {sfreq:.0f} Hz)"
)
print("Referências de ciclo:", BAND_PERIOD_MS)
df_phys


Faixa fisiológica sugerida: L_M = 13–25 pts (52–100 ms a 250 Hz)
Referências de ciclo: {'beta (~20 Hz)': 50.0, 'alpha (~10 Hz)': 100.0, 'theta (~8 Hz)': 125.0}


,L_M,L_M_ms,frac_alpha,frac_beta,faixa
0,5,20.0,0.20,0.40,curta demais (< ~1 ciclo beta)
1,6,24.0,0.24,0.48,curta demais (< ~1 ciclo beta)
2,7,28.0,0.28,0.56,curta demais (< ~1 ciclo beta)
3,8,32.0,0.32,0.64,curta demais (< ~1 ciclo beta)
4,9,36.0,0.36,0.72,curta demais (< ~1 ciclo beta)
5,10,40.0,0.40,0.80,curta demais (< ~1 ciclo beta)
6,11,44.0,0.44,0.88,curta demais (< ~1 ciclo beta)
7,12,48.0,0.48,0.96,curta demais (< ~1 ciclo beta)
8,13,52.0,0.52,1.04,compatível com alpha/beta (~50–150 ms)
9,14,56.0,0.56,1.12,compatível com alpha/beta (~50–150 ms)


In [72]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_phys["L_M"],
        y=df_phys["L_M_ms"],
        mode="lines+markers",
        name="Duração da janela",
        marker=dict(size=8),
    )
)
for name, period in BAND_PERIOD_MS.items():
    fig.add_hline(y=period, line_dash="dash", annotation_text=name)

fig.update_layout(
    title="Duração de L_M (ms) vs períodos de bandas EEG",
    xaxis_title="Tamanho da janela L_M (pts)",
    yaxis_title="Duração (ms)",
    template="plotly_white",
    hovermode="x unified",
    height=450,
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


### Critério 2b — IAF (pico alfa individual)

Estima a frequência alfa individual (IAF) em **todos** os registros **EC** disponíveis
em `data/preprocessed`, nos canais posteriores. Para a PSD, usa-se de preferência o trecho
antes do primeiro `boundary` (com fallback para o registro completo se esse trecho for curto).

A janela sugerida corresponde a **~1 ciclo de alpha** na mediana da amostra:
\(L_M = \mathrm{round}(f_s / f_{\alpha})\).


In [73]:
# ==========================
# IAF — listar todos os EC
# ==========================
files_ec = sorted(
    [p for p in files if p.stem.endswith("_EC")],
    key=lambda p: p.name,
)
print(f"Arquivos EC para IAF: {len(files_ec)}")
files_ec[:5]


Arquivos EC para IAF: 157


[WindowsPath('../../data/preprocessed/sub-010061_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010062_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010063_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010064_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010065_EC.set')]

In [74]:
# ==========================
# IAF — PSD Welch (8–12 Hz) em todos os EC
# ==========================
POSTERIORES = {"O1", "OZ", "O2", "PO3", "POZ", "PO4", "P3", "PZ", "P4"}
MIN_TMAX_S = 4.0  # se o trecho pré-boundary for curto demais, usa o registro inteiro

rows_iaf = []
errors_iaf = []

for file_ec in tqdm(files_ec, desc="IAF (EC)"):
    sid = file_ec.stem.split("_", 1)[0]
    try:
        raw_ec = mne.io.read_raw_eeglab(file_ec, preload=True, verbose=False)
        raw_ec.pick("eeg")
        sfreq_i = float(raw_ec.info["sfreq"])
        full_tmax = float(raw_ec.times[-1])

        boundary_onsets = [
            ann["onset"]
            for ann in raw_ec.annotations
            if str(ann["description"]).lower() == "boundary"
        ]
        tmax_boundary = min(boundary_onsets) if boundary_onsets else full_tmax
        tmax = tmax_boundary
        used_full = False
        if tmax < MIN_TMAX_S:
            tmax = full_tmax
            used_full = True
        if tmax < MIN_TMAX_S:
            raise ValueError(f"registro muito curto ({tmax:.2f} s)")

        posteriores = [ch for ch in raw_ec.ch_names if ch.upper() in POSTERIORES]
        picks = posteriores if posteriores else "eeg"

        n_samples = int(tmax * sfreq_i)
        n_per_seg = min(int(4 * sfreq_i), n_samples)
        n_per_seg = max(int(sfreq_i), n_per_seg)
        n_per_seg = min(n_per_seg, n_samples)
        n_overlap = min(n_per_seg // 2, max(0, n_per_seg - 1))

        spectrum = raw_ec.compute_psd(
            method="welch",
            fmin=8,
            fmax=12,
            tmin=0,
            tmax=tmax,
            picks=picks,
            n_per_seg=n_per_seg,
            n_overlap=n_overlap,
            verbose=False,
        )
        psd, freqs = spectrum.get_data(return_freqs=True)
        potencia_media = psd.mean(axis=0)
        iaf_i = float(freqs[int(np.argmax(potencia_media))])
        periodo_i = 1000.0 / iaf_i

        rows_iaf.append(
            {
                "subject": sid,
                "file": file_ec.name,
                "sfreq": sfreq_i,
                "tmax_s": tmax,
                "tmax_boundary_s": tmax_boundary,
                "used_full_recording": used_full,
                "n_posteriores": 0 if picks == "eeg" else len(posteriores),
                "iaf_hz": iaf_i,
                "periodo_ms": periodo_i,
                "L_M_pts": int(round(sfreq_i / iaf_i)),
                "WINDOW_MS": periodo_i,
            }
        )
    except Exception as exc:  # noqa: BLE001 — registrar e seguir
        errors_iaf.append({"subject": sid, "file": file_ec.name, "error": str(exc)})

df_iaf = pd.DataFrame(rows_iaf).sort_values("subject").reset_index(drop=True)

# opcional: persistir
out_csv = Path("../../data/intermediate/iaf_all_ec.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
df_iaf.to_csv(out_csv, index=False)

print(f"OK: {len(df_iaf)} | falhas: {len(errors_iaf)}")
if errors_iaf:
    display(pd.DataFrame(errors_iaf))

iaf_hz = float(df_iaf["iaf_hz"].median())
periodo_ms = float(df_iaf["periodo_ms"].median())
L_iaf = int(round(df_iaf["L_M_pts"].median()))
WINDOW_MS_IAF = periodo_ms
sfreq_iaf = float(df_iaf["sfreq"].mode().iloc[0]) if len(df_iaf) else float("nan")

print(
    f"\nIAF mediana: {iaf_hz:.2f} Hz "
    f"(IQR {df_iaf['iaf_hz'].quantile(0.25):.2f}–{df_iaf['iaf_hz'].quantile(0.75):.2f})"
)
print(f"Período mediana: {periodo_ms:.1f} ms")
print(
    f"Janela sugerida (mediana): L_M={L_iaf} pts | WINDOW_MS ≈ {WINDOW_MS_IAF:.0f} ms"
)
print(f"Fallback registro completo (<{MIN_TMAX_S:.0f}s pré-boundary): "
      f"{int(df_iaf['used_full_recording'].sum())}")
df_iaf.describe(include="all")


IAF (EC):   0%|          | 0/157 [00:00<?, ?it/s]C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2807788333.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2807788333.py:13: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.

IAF (EC):   1%|          | 1/157 [00:00<00:56,  2.77it/s]C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2807788333.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2807788333.py:13: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.

IAF (EC):   1%|▏         | 2/157 [00:00<01:03,  2.42it/s]C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2807788333.py:13: RuntimeWarning:

Limited 1 annotatio

OK: 157 | falhas: 0

IAF mediana: 9.89 Hz (IQR 9.28–10.62)
Período mediana: 101.1 ms
Janela sugerida (mediana): L_M=25 pts | WINDOW_MS ≈ 101 ms
Fallback registro completo (<4s pré-boundary): 19


,subject,file,sfreq,tmax_s,tmax_boundary_s,used_full_recording,n_posteriores,iaf_hz,periodo_ms,L_M_pts,WINDOW_MS
count,157,157,157.000000,157.000000,157.000000,157,157.000000,157.000000,157.000000,157.000000,157.000000
unique,157,157,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN
top,sub-010061,sub-010061_EC.set,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
freq,1,1,NaN,NaN,NaN,138,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,248.089172,108.008803,50.393987,NaN,8.987261,9.893138,101.936800,25.324841,101.936800
std,NaN,NaN,16.875629,137.325252,20.702119,NaN,0.112504,0.908111,9.454908,2.833446,9.454908
min,NaN,NaN,100.000000,14.502000,0.002000,NaN,8.000000,8.007812,83.591837,10.000000,83.591837
25%,NaN,NaN,250.000000,59.982000,59.690000,NaN,9.000000,9.277344,94.160920,24.000000,94.160920
50%,NaN,NaN,250.000000,59.998000,59.998000,NaN,9.000000,9.887695,101.135802,25.000000,101.135802
75%,NaN,NaN,250.000000,59.998000,59.998000,NaN,9.000000,10.620117,107.789474,27.000000,107.789474


In [81]:

df_iaf[["subject", "iaf_hz", "periodo_ms", "L_M_pts", "WINDOW_MS"]].head(10)


,subject,iaf_hz,periodo_ms,L_M_pts,WINDOW_MS
0,sub-010061,10.498047,95.255814,24,95.255814
1,sub-010062,10.742188,93.090909,23,93.090909
2,sub-010063,9.521484,105.025641,26,105.025641
3,sub-010064,9.155273,109.226667,27,109.226667
4,sub-010065,9.033203,110.702703,28,110.702703
5,sub-010066,9.033203,110.702703,28,110.702703
6,sub-010067,9.643555,103.696203,26,103.696203
7,sub-010068,9.887695,101.135802,25,101.135802
8,sub-010069,9.033203,110.702703,28,110.702703
9,sub-010070,11.108398,90.021978,23,90.021978


In [76]:
from plotly.subplots import make_subplots

fig_tcc = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "(a) Distribuição da frequência alfa individual",
        "(b) Distribuição do período alfa",
    ),
    vertical_spacing=0.16,
)

# Aproveita os histogramas que você já criou
fig_tcc.add_trace(
    go.Histogram(x=df_iaf["iaf_hz"], nbinsx=20, marker_color="steelblue", name="IAF"),
    row=1, col=1,
)
fig_tcc.add_trace(
    go.Histogram(x=df_iaf["periodo_ms"], nbinsx=20, marker_color="seagreen", name="Período"),
    row=1, col=2,
)
fig_tcc.add_vline(
    x=float(iaf_hz), row=1, col=1,
    line_dash="dash", line_color="crimson",
    annotation_text=f"mediana = {iaf_hz:.2f} Hz",
    annotation_position="top",
)
fig_tcc.add_vline(
    x=float(periodo_ms), row=1, col=2,
    line_dash="dash", line_color="crimson",
    annotation_text=f"mediana = {periodo_ms:.0f} ms",
    annotation_position="top",
)
fig_tcc.update_xaxes(title_text="IAF (Hz)", title_standoff=12, row=1, col=1)
fig_tcc.update_xaxes(title_text="Período alfa (ms)", title_standoff=12, row=1, col=2)
fig_tcc.update_yaxes(title_text="Contagem", title_standoff=18, row=1, col=1)
fig_tcc.update_yaxes(title_text="Contagem", title_standoff=18, row=1, col=2)
fig_tcc.update_layout(
    height=420,
    width=1000,
    template="plotly_white",
    showlegend=False,
    margin=dict(t=110, l=70, r=30, b=60),  
)
# 1) Títulos dos painéis alinhados à esquerda
for ann in fig_tcc.layout.annotations:
    if ann.text.startswith("(a)") or ann.text.startswith("(b)"):
        ann.xanchor = "left"
        ann.yshift =30

        if ann.text.startswith("(a)"):
            ann.x = fig_tcc.layout.xaxis.domain[0]
        else:
            ann.x = fig_tcc.layout.xaxis2.domain[0]
fig_tcc.show()

A IAF foi estimada em todos os registros EC disponíveis. A **mediana** da amostra é usada
como referência fisiológica para a janela (~1 ciclo alfa); os histogramas acima resumem
a dispersão entre participantes.


### Resumo — \(L_M\) sugerido pelos critérios adotados

Consolida o critério de arestas (Rosário) e o critério fisiológico (faixa de bandas + IAF).


In [77]:
# ==========================
# RESUMO — L_M (critérios adotados)
# ==========================
sep = (
    (df_na["Na_original"] > 2 * df_na["Na_embaralhado"])
    & ((df_na["Na_original"] - df_na["Na_embaralhado"]) >= 10)
)
if sep.any():
    L_rosario = int(df_na.loc[sep].iloc[0]["L_M"])
    nota_rosario = "menor L_M com separação Na_orig vs Na_shuf (piso)"
else:
    L_rosario = None
    nota_rosario = "sem separação clara na faixa"

resumo = [
    {
        "metodo": "1. Arestas (orig vs embaralhada)",
        "L_M_pts": L_rosario,
        "WINDOW_MS": None if L_rosario is None else L_rosario / sfreq * 1000,
        "nota": nota_rosario,
    },
    {
        "metodo": "2. Fisiológico (faixa alpha/beta)",
        "L_M_pts": f"{L_phys_min}–{L_phys_max}",
        "WINDOW_MS": f"{L_phys_min / sfreq * 1000:.0f}–{L_phys_max / sfreq * 1000:.0f}",
        "nota": "duração ~50–150 ms (alpha/beta)",
    },
    {
        "metodo": "2b. IAF (~1 ciclo alfa)",
        "L_M_pts": L_iaf,
        "WINDOW_MS": WINDOW_MS_IAF,
        "nota": (
            f"mediana IAF {iaf_hz:.2f} Hz "
            f"(n={len(df_iaf)} EC; IQR "
            f"{df_iaf['iaf_hz'].quantile(0.25):.2f}–"
            f"{df_iaf['iaf_hz'].quantile(0.75):.2f})"
        ),
    },
]

df_resumo = pd.DataFrame(resumo)
df_resumo


,metodo,L_M_pts,WINDOW_MS,nota
0,1. Arestas (orig vs embaralhada),8,32.0,menor L_M com separação Na_orig vs Na_shuf (piso)
1,2. Fisiológico (faixa alpha/beta),13–25,52–100,duração ~50–150 ms (alpha/beta)
2,2b. IAF (~1 ciclo alfa),25,101.135802,mediana IAF 9.89 Hz (n=157 EC; IQR 9.28–10.62)


O critério de arestas indica o **piso** de \(L_M\) a partir do qual a rede original se
distingue da nula. O critério fisiológico (faixa alpha/beta e IAF mediana da amostra EC)
aponta para uma janela da ordem de **~96–100 ms** (~1 ciclo alfa).

Assim, foi adotado **100 ms** (\(L_M = 25\) a 250 Hz): valor arredondado, reprodutível e
compatível com a escala temporal da atividade alfa, respeitando o piso do critério de arestas.


## Trecho contínuo comum para o TVG

Esta seção **não** define a IAF. O objetivo é escolher o maior intervalo de tempo $[t_0, t_1]$
(relativo ao início de cada arquivo `.set`) em que **todos** os registros EO e EC do pipeline
têm sinal **contínuo** — isto é, sem evento `boundary` no meio.

Os `boundary` do EEGLAB marcam descontinuidades. Cada arquivo é partido em segmentos contínuos
entre boundaries; a cobertura do arquivo é a união desses segmentos. A cobertura **comum** é a
interseção das coberturas de todos os arquivos; o resultado é o **maior intervalo contíguo**
dessa interseção.

Exemplo: participante A contínuo em $[2, 10]$ s e B em $[1, 8]$ s → trecho comum $[2, 8]$ s
(duração 6 s). Esse $[t_0, t_1]$ pode ser usado como recorte compartilhado no TVG
O pipeline (`01_build_networks`) usa `N_SEC` segundos a partir de logo após `t0`,
com checagem de que o recorte cabe em `[t0, t1]` e não atravessa `boundary`.


In [83]:
# ==========================
# Trecho contínuo comum (EO + EC) para o TVG
# ==========================
import json
from pathlib import Path

OUT_DIR = Path("../../data/intermediate")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "tvg_common_continuous.csv"
OUT_JSON = OUT_DIR / "tvg_common_continuous_summary.json"

files_tvg = sorted(
    [p for p in files if p.stem.endswith("_EO") or p.stem.endswith("_EC")],
    key=lambda p: p.name,
)
print(f"Arquivos EO/EC para trecho comum: {len(files_tvg)}")


def continuous_segments(raw) -> list[tuple[float, float]]:
    """Segmentos contínuos [start, end] entre boundaries (e extremos do registro)."""
    tmax = float(raw.times[-1])
    bounds = sorted(
        {
            float(ann["onset"])
            for ann in raw.annotations
            if str(ann["description"]).lower() == "boundary"
            and 0.0 < float(ann["onset"]) < tmax
        }
    )
    cuts = [0.0] + bounds + [tmax]
    segs = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        if b > a:
            segs.append((a, b))
    return segs


def longest_segment(segs: list[tuple[float, float]]) -> tuple[float, float] | None:
    if not segs:
        return None
    return max(segs, key=lambda ab: ab[1] - ab[0])


rows_seg = []
errors_seg = []
all_covers: list[list[tuple[float, float]]] = []

for fpath in tqdm(files_tvg, desc="Segmentos contínuos"):
    sid, cond = fpath.stem.split("_", 1)
    try:
        raw = mne.io.read_raw_eeglab(fpath, preload=False, verbose=False)
        # só metadados/tempos/annotations — sem preload completo
        tmax = float(raw.times[-1])
        segs = continuous_segments(raw)
        n_bound = sum(
            1
            for ann in raw.annotations
            if str(ann["description"]).lower() == "boundary"
        )
        longest = longest_segment(segs)
        rows_seg.append(
            {
                "subject": sid,
                "condition": cond,
                "file": fpath.name,
                "duration_s": tmax,
                "n_boundaries": n_bound,
                "n_segments": len(segs),
                "segments_json": json.dumps(segs),
                "longest_start_s": None if longest is None else longest[0],
                "longest_end_s": None if longest is None else longest[1],
                "longest_dur_s": None if longest is None else longest[1] - longest[0],
            }
        )
        all_covers.append(segs)
    except Exception as exc:  # noqa: BLE001
        errors_seg.append({"file": fpath.name, "error": str(exc)})

df_seg = pd.DataFrame(rows_seg).sort_values(["subject", "condition"]).reset_index(drop=True)
print(f"OK: {len(df_seg)} | falhas: {len(errors_seg)}")
if errors_seg:
    display(pd.DataFrame(errors_seg))
df_seg.head()


Arquivos EO/EC para trecho comum: 314


Segmentos contínuos:   0%|          | 0/314 [00:00<?, ?it/s]

C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2164226712.py:51: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2164226712.py:51: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.

C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2164226712.py:51: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2164226712.py:51: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.

Segmentos contínuos:   1%|          | 2/314 [00:00<00:20, 15.56it/s]C:\Users\Anne\AppData\Local\Temp\ipykernel_21552\2164226712.py:51: RuntimeWarning:

Limited 2 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_21

OK: 314 | falhas: 0


,subject,condition,file,duration_s,n_boundaries,n_segments,segments_json,longest_start_s,longest_end_s,longest_dur_s
0,sub-010060,EO,sub-010060_EO.set,447.164,14,15,"[[0.0, 59.998], [59.998, 100.622], [100.622, 1...",223.258,283.258,60.0
1,sub-010061,EC,sub-010061_EC.set,470.828,8,9,"[[0.0, 43.382], [43.382, 50.83], [50.83, 110.8...",110.830,170.830,60.0
2,sub-010061,EO,sub-010061_EO.set,467.952,8,9,"[[0.0, 59.998], [59.998, 119.998], [119.998, 1...",59.998,119.998,60.0
3,sub-010062,EC,sub-010062_EC.set,473.156,8,9,"[[0.0, 0.002], [0.002, 55.822], [55.822, 115.7...",233.158,293.158,60.0
4,sub-010062,EO,sub-010062_EO.set,457.512,10,11,"[[0.0, 0.002], [0.002, 55.046], [55.046, 59.84...",288.294,348.294,60.0


In [84]:
# ==========================
# Interseção das coberturas → maior intervalo contíguo comum
# ==========================

def intersect_interval_lists(
    covers: list[list[tuple[float, float]]],
) -> list[tuple[float, float]]:
    """Interseção de N listas de intervalos (já união por arquivo = segmentos disjuntos)."""
    if not covers:
        return []
    common = list(covers[0])
    for other in covers[1:]:
        nxt: list[tuple[float, float]] = []
        for a0, a1 in common:
            for b0, b1 in other:
                lo, hi = max(a0, b0), min(a1, b1)
                if hi > lo:
                    nxt.append((lo, hi))
        # merge overlaps (por segurança)
        nxt.sort()
        merged: list[tuple[float, float]] = []
        for lo, hi in nxt:
            if not merged or lo >= merged[-1][1]:  # não fundir só no ponto do boundary
                merged.append((lo, hi))
            else:
                merged[-1] = (merged[-1][0], max(merged[-1][1], hi))
        common = merged
        if not common:
            break
    return common


common_intervals = intersect_interval_lists(all_covers)
if not common_intervals:
    raise RuntimeError(
        "Interseção vazia: não há intervalo [t0, t1] contínuo em comum a todos os arquivos."
    )

t0, t1 = max(common_intervals, key=lambda ab: ab[1] - ab[0])
duracao_s = t1 - t0
N_SEC_SUGERIDO = int(np.floor(duracao_s))

summary = {
    "n_files": int(len(df_seg)),
    "n_common_intervals": len(common_intervals),
    "t0": float(t0),
    "t1": float(t1),
    "duration_s": float(duracao_s),
    "N_SEC_sugerido": int(N_SEC_SUGERIDO),
    "common_intervals": [[float(a), float(b)] for a, b in common_intervals],
    "nota": (
        "Pipeline: crop_continuous_segment(raw, t0, t1, N_SEC) "
        "(início após t0; sem boundary no recorte)."
    ),
}

print(
    f"Maior trecho contínuo COMUM a todos: [{t0:.3f}, {t1:.3f}] s "
    f"(duração = {duracao_s:.3f} s)"
)
print(f"Intervalos comuns encontrados: {len(common_intervals)}")
print(f"Sugestão: N_SEC ≈ {N_SEC_SUGERIDO} s (se o recorte começar em t0={t0:.3f} s)")
print(
    f"Alternativa explícita: eeg[:, int({t0:.6f}*sfreq):int({t1:.6f}*sfreq)]"
)

# Verificação: [t0,t1] ⊆ algum segmento de cada arquivo
bad = []
for _, row in df_seg.iterrows():
    segs = json.loads(row["segments_json"])
    ok = any(a - 1e-9 <= t0 and t1 <= b + 1e-9 for a, b in segs)
    if not ok:
        bad.append(row["file"])
print(f"Validação [t0,t1] ⊆ segmento de cada arquivo: falhas={len(bad)}")
if bad:
    print("Arquivos fora:", bad[:10])

df_seg.to_csv(OUT_CSV, index=False)
OUT_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Salvo: {OUT_CSV.resolve()}")
print(f"Salvo: {OUT_JSON.resolve()}")
summary


Maior trecho contínuo COMUM a todos: [60.002, 68.274] s (duração = 8.272 s)
Intervalos comuns encontrados: 1263
Sugestão: N_SEC ≈ 8 s (se o recorte começar em t0=60.002 s)
Alternativa explícita: eeg[:, int(60.002000*sfreq):int(68.274000*sfreq)]
Validação [t0,t1] ⊆ segmento de cada arquivo: falhas=0
Salvo: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate\tvg_common_continuous.csv
Salvo: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate\tvg_common_continuous_summary.json


{'n_files': 314,
 'n_common_intervals': 1263,
 't0': 60.002,
 't1': 68.274,
 'duration_s': 8.271999999999998,
 'N_SEC_sugerido': 8,
 'common_intervals': [[0.0, 0.002],
  [0.002, 0.006],
  [0.006, 0.01],
  [0.01, 0.018],
  [0.018, 0.026],
  [0.026, 0.042],
  [0.042, 0.045],
  [0.045, 1.006],
  [1.006, 3.442],
  [3.442, 9.21],
  [9.21, 10.806],
  [10.806, 14.502],
  [14.502, 14.57],
  [14.57, 15.074],
  [15.074, 16.41],
  [16.41, 16.722],
  [16.722, 17.014],
  [17.014, 17.818],
  [17.818, 18.19],
  [18.19, 18.486],
  [18.486, 22.442],
  [22.442, 24.414],
  [24.414, 25.142],
  [25.142, 29.13],
  [29.13, 30.586],
  [30.586, 35.15],
  [35.15, 36.722],
  [36.722, 39.998],
  [39.998, 40.306],
  [40.306, 41.77],
  [41.77, 42.698],
  [42.698, 43.382],
  [43.382, 43.478],
  [43.478, 44.05],
  [44.05, 44.898],
  [44.898, 45.622],
  [45.622, 47.47],
  [47.47, 49.326],
  [49.326, 49.37],
  [49.37, 49.51],
  [49.51, 50.83],
  [50.83, 51.17],
  [51.17, 51.498],
  [51.498, 52.15],
  [52.15, 52.334],
 

In [85]:
# Maior segmento individual vs trecho comum
# (histograma no range DOS DADOS; a linha "comum" só faz sentido se <= max individual)
vals = df_seg["longest_dur_s"].dropna().astype(float)
print(
    f"Maior segmento individual: min={vals.min():.2f}s | "
    f"mediana={vals.median():.2f}s | max={vals.max():.2f}s | n={len(vals)}"
)
print(f"Trecho comum (duração): {duracao_s:.2f} s  em [{t0:.2f}, {t1:.2f}] s")
if duracao_s > vals.max() + 1e-6:
    print(
        "AVISO: duração comum > maior segmento individual — "
        "rode de novo a célula da interseção (merge de intervals)."
    )

fig_c = go.Figure()
fig_c.add_trace(
    go.Histogram(
        x=vals,
        xbins=dict(
            start=float(vals.min()) - 0.5,
            end=float(vals.max()) + 0.5,
            size=max(1.0, (float(vals.max()) - float(vals.min())) / 20.0),
        ),
        name="Maior segmento por arquivo",
        marker=dict(color="steelblue"),
    )
)
# linha do comum só se estiver na escala dos dados (senão anota fora)
if duracao_s <= float(vals.max()) * 1.05:
    fig_c.add_vline(
        x=duracao_s,
        line_dash="dash",
        line_color="crimson",
        annotation_text=f"comum = {duracao_s:.2f} s",
    )

fig_c.update_layout(
    title=(
        f"Duração do maior segmento contínuo por arquivo (EO/EC) — n={len(vals)}<br>"
        f"<sup>Trecho comum a todos: [{t0:.2f}, {t1:.2f}] s "
        f"(duração {duracao_s:.2f} s)</sup>"
    ),
    xaxis_title="Duração do maior segmento (s)",
    yaxis_title="Contagem",
    template="plotly_white",
    height=420,
    bargap=0.05,
)
fig_c.update_xaxes(range=[float(vals.min()) - 1, float(vals.max()) + 1])
display(HTML(fig_c.to_html(full_html=False, include_plotlyjs="cdn")))

df_seg.nsmallest(10, "longest_dur_s")[
    ["subject", "condition", "file", "longest_start_s", "longest_end_s", "longest_dur_s"]
]


Maior segmento individual: min=59.62s | mediana=60.00s | max=60.00s | n=314
Trecho comum (duração): 8.27 s  em [60.00, 68.27] s


,subject,condition,file,longest_start_s,longest_end_s,longest_dur_s
100,sub-010168,EO,sub-010168_EO.set,275.826,335.450,59.624
235,sub-010272,EO,sub-010272_EO.set,222.826,282.506,59.680
247,sub-010283,EO,sub-010283_EO.set,230.986,290.778,59.792
219,sub-010264,EO,sub-010264_EO.set,331.602,391.450,59.848
281,sub-010302,EO,sub-010302_EO.set,252.690,312.574,59.884
106,sub-010183,EO,sub-010183_EO.set,388.790,448.680,59.890
128,sub-010204,EO,sub-010204_EO.set,260.790,320.690,59.900
124,sub-010201,EO,sub-010201_EO.set,117.658,177.578,59.920
6,sub-010063,EO,sub-010063_EO.set,0.000,59.998,59.998
49,sub-010089,EC,sub-010089_EC.set,0.000,59.998,59.998
